In [1]:
import nibabel as nib
import numpy as np
from pathlib import Path

data_dir = Path("data/Merlin Abdominal CT Dataset_files")
files = [f for f in sorted(data_dir.glob("*.nii.gz"))]

print(files)
# img = nib.load(files[0])
# volume = img.get_fdata()

# print(volume)
# print(volume.shape)
# print(volume.dtype)

# print(volume.min())
# print(volume.max())

# img.header.get_zooms()

[PosixPath('data/Merlin Abdominal CT Dataset_files/AC421363e.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC421363f.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213641.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213644.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213645.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213646.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213647.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213649.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC421364a.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC421364b.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC421364d.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC421364e.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC421364f.nii.gz'), PosixPath('data/Merlin Abdominal CT Dataset_files/AC4213650.nii.gz'), PosixPath('data/Mer

In [24]:
FILTERED_FINDING_PHRASES = {
    "renal_cyst":                     ("a renal cyst is present", "no renal cyst"),
    "surgically_absent_gallbladder":  ("the gallbladder is surgically absent", "the gallbladder is present"),
    "atelectasis":                    ("atelectasis is present", "no atelectasis"),
    "pleural_effusion":               ("a pleural effusion is present", "no pleural effusion"),
}

In [ ]:
from merlin.data import DataLoader
from merlin import Merlin
import torch
import torch.nn.functional as F
import json
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

datalist = [{"image": f, "study_id": f.stem.removesuffix(".nii")} for f in files]
# datalist = [{"image": files[0], "study_id": "AC421363e"}, {"image": files[1], "study_id": "AC421363f"}]

dataloader = DataLoader(
    datalist=datalist,
    cache_dir=None,
    batchsize=1,
    shuffle=True,
    num_workers=0,
)

# model_img = Merlin(ImageEmbedding=True)
# model_img.eval().cuda()

model = Merlin()
model.eval().cuda()

flat_phrases, phrase_keys = [], []

for finding, (present, absence) in FILTERED_FINDING_PHRASES.items():
    flat_phrases += [present, absence]
    phrase_keys += [(finding, "present"), (finding, "absent")]

img_embeds = {}
dummy_tensor = None
with torch.no_grad():
    for batch in dataloader:
        sid = batch["study_id"][0]
        if dummy_tensor is None:
            dummy_tensor = batch["image"].cuda()
        img_out, _, _ = model(batch["image"].cuda(), flat_phrases)
        img_embeds[sid] = img_out[0]

with torch.no_grad():
    _, _, text_embeds = model(dummy_tensor, flat_phrases)

text_lookup = {}
for i, (finding, polarity) in enumerate(phrase_keys):
    text_lookup.setdefault(finding, {})[polarity] = text_embeds[i]

labels = pd.read_csv("data/zero_shot_findings_disease_cls.csv").set_index("study_id")

results = {}

with torch.no_grad():
    for finding in FILTERED_FINDING_PHRASES:
        sid = list(img_embeds.keys())

        sid_list, y_true, y_score = [], [], []

        for sid_i in sid:
            label = labels.loc[sid_i, finding]
            if label == -1:
                continue

            sim_present = F.cosine_similarity(img_embeds[sid_i], text_lookup[finding]["present"], dim=0)
            sim_absent = F.cosine_similarity(img_embeds[sid_i], text_lookup[finding]["absent"], dim=0)

            probs = F.softmax(torch.stack([sim_present, sim_absent]), dim=0)
            score = probs[0].item()

            sid_list.append(sid_i)
            y_score.append(score)
            y_true.append(int(label))

        results[finding] = {"study_id": sid_list, "y_true": y_true, "y_score": y_score}

        
        with open("merlin_zero_shot_results.json", "w") as f:
            json.dump(results, f, indent=2)

In [5]:
import json
from sklearn.metrics import roc_auc_score

with open("merlin_zero_shot_results.json") as f:
    results = json.load(f)

for finding, d in results.items():
    y_true, y_score = d["y_true"], d["y_score"]

    print(f"{finding}: n={len(y_true)}, positives={sum(y_true)}")

    # auroc = roc_auc_score(y_true, y_score)
    # print(auroc)

renal_cyst: n=38, positives=10
surgically_absent_gallbladder: n=53, positives=14
atelectasis: n=41, positives=20
pleural_effusion: n=28, positives=9


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

for finding, d in results.items():
    y_true = d["y_true"]
    y_pred = [1 if s >= 0.5 else 0 for s in d["y_score"]]   # threshold at 0.5

    print(finding,
          "F1:", f1_score(y_true, y_pred),
          "precision:", precision_score(y_true, y_pred, zero_division=0),
          "recall:", recall_score(y_true, y_pred, zero_division=0))

renal_cyst F1: 0.21052631578947367 precision: 0.14285714285714285 recall: 0.4
surgically_absent_gallbladder F1: 0.75 precision: 0.6666666666666666 recall: 0.8571428571428571
atelectasis F1: 0.5333333333333333 precision: 0.8 recall: 0.4
pleural_effusion F1: 0.9 precision: 0.8181818181818182 recall: 1.0


In [9]:
import json
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# Load results
with open("merlin_zero_shot_results.json") as f:
    results = json.load(f)

# Define column widths for neat table formatting
header = f"{'Finding':<32} | {'N':<6} | {'Pos':<5} | {'AUROC':<7} | {'F1':<7} | {'Precision':<7} | {'Recall':<7}"
divider = "-" * len(header)

print(header)
print(divider)

# Tracking metrics for overall macro-average
auroc_list, f1_list, prec_list, rec_list = [], [], [], []

for finding, d in results.items():
    y_true, y_score = d["y_true"], d["y_score"]
    n_samples = len(y_true)
    n_positives = sum(y_true)

    # Threshold at 0.5 for binary classification metrics
    y_pred = [1 if s >= 0.5 else 0 for s in y_score]

    # Calculate metrics
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)

    # Handle single-class edge cases for AUROC
    if len(set(y_true)) < 2:
        auroc_str = "N/A*"
        print(f"{finding:<32} | {n_samples:<6} | {n_positives:<5} | {auroc_str:<7} | {f1:.4f}  | {prec:.4f}  | {rec:.4f}")
        continue

    auroc = roc_auc_score(y_true, y_score)
    
    # Store for summary stats
    auroc_list.append(auroc)
    f1_list.append(f1)
    prec_list.append(prec)
    rec_list.append(rec)

    # Print formatted row
    print(f"{finding:<32} | {n_samples:<6} | {n_positives:<5} | {auroc:.4f}  | {f1:.4f}  | {prec:.4f}  | {rec:.4f}")

# Print Macro Averages
print(divider)
if auroc_list:
    macro_auroc = sum(auroc_list) / len(auroc_list)
    macro_f1 = sum(f1_list) / len(f1_list)
    macro_prec = sum(prec_list) / len(prec_list)
    macro_rec = sum(rec_list) / len(rec_list)
    
    print(f"{'MACRO AVERAGE':<32} | {'-':<6} | {'-':<5} | {macro_auroc:.4f}  | {macro_f1:.4f}  | {macro_prec:.4f}  | {macro_rec:.4f}")
print(divider)

Finding                          | N      | Pos   | AUROC   | F1      | Precision | Recall 
-------------------------------------------------------------------------------------------
renal_cyst                       | 38     | 10    | 0.3179  | 0.2105  | 0.1429  | 0.4000
surgically_absent_gallbladder    | 53     | 14    | 0.9231  | 0.7500  | 0.6667  | 0.8571
atelectasis                      | 41     | 20    | 0.8524  | 0.5333  | 0.8000  | 0.4000
pleural_effusion                 | 28     | 9     | 0.9649  | 0.9000  | 0.8182  | 1.0000
-------------------------------------------------------------------------------------------
MACRO AVERAGE                    | -      | -     | 0.7646  | 0.5985  | 0.6069  | 0.6643
-------------------------------------------------------------------------------------------
